# Partial O3 with ranking

Compute CAM/WACCM hybrid-level partial O3 columns for these cases:

- `/mnt/soclim0/public_data/weiji/B2000WCN001002_timefixed`
- `/mnt/soclim0/public_data/weiji/B2000WCN_NOCOUPL001002_timefixed`
- `/mnt/soclim0/public_data/weiji/BWCN`

Outputs go into each case directory under `partial_O3`.

Each experiment writes one aggregate NetCDF file containing all requested pressure ranges (`30-70 hPa`, `1-100 hPa`, `30-100 hPa`) across all years. Ranking variables are stored in the same NetCDF file and are also exported as a compact CSV table. The calculation uses interface-pressure overlap on hybrid levels, so partial layers at the pressure bounds are handled exactly instead of selecting whole model levels.


In [ ]:
from pathlib import Path
import csv
import re

import dask
import numpy as np
import xarray as xr

try:
    from dask.diagnostics import ProgressBar
except Exception:
    ProgressBar = None

BASE_DIR = Path("/mnt/soclim0/public_data/weiji")

CASES = {
    "B2000WCN": BASE_DIR / "B2000WCN001002_timefixed",
    "B2000WCN_NOCOUPL": BASE_DIR / "B2000WCN_NOCOUPL001002_timefixed",
    "B2000WCN007009010011_Clim3D": BASE_DIR / "B2000WCN007009010011_Clim3D_timefixed",
    "BWCN": BASE_DIR / "BWCN",
}

PRESSURE_RANGES = [
    (30, 70, "30_70hPa"),
    (1, 100, "1_100hPa"),
    (30, 100, "30_100hPa"),
]

# Ranking follows Friedel et al.-style ozone-year selection: polar-cap mean,
# 5-day running mean, then annual March-April minimum/maximum by event year.
# Prefer CAM's integer date variable (YYYYMMDD); fall back to no-leap time if date is incomplete.
POLAR_LAT_BOUNDS = (60, 90)
RANKING_MONTHS = [3, 4]
RANKING_ROLLING_DAYS = 5
RANKING_ROLLING_CENTER = True
RANKING_ROLLING_MIN_PERIODS = 5
RANKING_MIN_VALID_DAYS = 58
RANKING_MIN_VALID_DU = 10.0

# Ranking CSV contains both definitions:
#   raw : Mar-Apr extrema of unsmoothed daily partial O3
#   rm5 : Mar-Apr extrema after 5-day running mean
RANKING_METHODS = {
    "raw": {"rolling_days": 1, "center": False, "min_periods": 1},
    "rm5": {"rolling_days": 5, "center": True, "min_periods": 5},
}

# Keep dask chunks modest. A full-year/full-grid/full-level O3 chunk is ~1.3 GB and can crash kernels.
CHUNKS = {"time": 30, "lev": 66, "lat": 48, "lon": 72}
OUTPUT_CHUNKS = {"time": 30, "lat": 48, "lon": 72}
DASK_SCHEDULER = "single-threaded"

# Re-run policy. Existing readable aggregate files are skipped unless OVERWRITE=True.
# Corrupt/incomplete aggregate files are rebuilt automatically.
OVERWRITE = False

# Each experiment writes one aggregate NetCDF containing all pressure ranges and ranking variables.
MAKE_AGGREGATE_NC = True
MAKE_RANKINGS = True

# If True, do not recompute partial O3 and do not append NetCDF ranking variables;
# only rebuild/overwrite the ranking CSV from the existing aggregate NetCDF.
# This is useful after changing ranking rules while expensive partial-O3 files already exist.
RANKING_CSV_ONLY = False
RANKING_CSV_ONLY_SKIP_MISSING = True
OVERWRITE_RANKING_CSV = True

# netCDF4 enables zlib compression. If unavailable, save_dataset retries without compression.
NETCDF_ENGINE = "netcdf4"
USE_PROGRESS_BAR = True


In [2]:
# Constants for mol/mol O3 -> Dobson Units via dp integration.
G = 9.80665
M_AIR = 28.964 / 1000.0
NA = 6.02214e23
DU_MOLECULES_PER_M2 = 2.687e20
DU_FACTOR = NA / (G * M_AIR * DU_MOLECULES_PER_M2)

YEAR_RE = re.compile(r"\.h3\.(\d{4})\.O3\.nc$")


def extract_h3_year(path):
    match = YEAR_RE.search(Path(path).name)
    if match is None:
        return None
    return int(match.group(1))


def discover_o3_files(case_dir):
    case_dir = Path(case_dir)
    files = list((case_dir / "O3").glob("*.O3.nc"))
    return sorted(
        files,
        key=lambda p: (extract_h3_year(p) if extract_h3_year(p) is not None else 10**9, p.name),
    )


def find_matching_ps_file(case_dir, o3_file):
    year = extract_h3_year(o3_file)
    if year is None:
        return None
    candidates = sorted((Path(case_dir) / "PS").glob(f"*.h3.{year:04d}.PS.nc"))
    return candidates[0] if candidates else None


def open_o3_dataset(case_dir, o3_file):
    ds = xr.open_dataset(o3_file, chunks=CHUNKS, decode_times=False)

    # Current O3 files already include PS/P0/hyai/hybi. This fallback keeps the notebook
    # usable if a future split file only contains O3.
    if "PS" not in ds:
        ps_file = find_matching_ps_file(case_dir, o3_file)
        if ps_file is None:
            raise FileNotFoundError(f"No matching PS file found for {o3_file}")
        ds_ps = xr.open_dataset(ps_file, chunks=CHUNKS, decode_times=False)
        keep = [v for v in ["PS", "P0", "hyai", "hybi", "hyam", "hybm", "lev", "ilev", "date", "datesec", "time_bnds", "gw"] if v in ds_ps]
        ds = xr.merge([ds, ds_ps[keep]], compat="override")

    required = ["O3", "P0", "PS", "hyai", "hybi"]
    missing = [name for name in required if name not in ds]
    if missing:
        raise KeyError(f"Missing required variables in {o3_file}: {missing}")
    if "lev" not in ds.dims and "lev" not in ds.coords:
        raise KeyError(f"Missing lev coordinate/dimension in {o3_file}")

    return ds


def open_case_dataset(case_dir, files):
    """Open all yearly O3 files for one experiment as one dask-backed dataset."""
    if not files:
        raise FileNotFoundError(f"No O3 files found in {Path(case_dir) / 'O3'}")

    ds = xr.open_mfdataset(
        [str(f) for f in files],
        combine="nested",
        concat_dim="time",
        parallel=False,
        chunks=CHUNKS,
        decode_times=False,
        data_vars="minimal",
        coords="minimal",
        compat="override",
    )

    required = ["O3", "P0", "PS", "hyai", "hybi", "date"]
    missing = [name for name in required if name not in ds]
    if missing:
        raise KeyError(f"Missing required variables after opening {case_dir}: {missing}")
    return ds


def case_output_file(case_name, out_dir):
    return Path(out_dir) / f"{case_name}_partial_O3_all_ranges.nc"


def is_valid_netcdf(path):
    path = Path(path)
    if not path.exists():
        return False
    try:
        with xr.open_dataset(path, decode_times=False) as ds:
            _ = tuple(ds.dims)
        return True
    except Exception as exc:
        print(f"Existing NetCDF is not readable and will be rebuilt: {path}")
        print(f"  Reason: {exc}")
        return False


def build_hybrid_overlap_dp(ds_sub, p_top_hpa, p_bot_hpa):
    """Return overlap pressure thickness in Pa on each hybrid layer."""
    if p_top_hpa >= p_bot_hpa:
        raise ValueError("p_top_hpa must be smaller than p_bot_hpa")

    P0 = ds_sub["P0"]
    PS = ds_sub["PS"]

    P_interface = ds_sub["hyai"] * P0 + ds_sub["hybi"] * PS

    p_i = P_interface.isel(ilev=slice(0, -1)).rename({"ilev": "lev"})
    p_ip1 = P_interface.isel(ilev=slice(1, None)).rename({"ilev": "lev"})

    # Force the same lev coordinate as O3, avoiding xarray alignment surprises.
    if "lev" in ds_sub.coords:
        p_i = p_i.assign_coords(lev=ds_sub["lev"])
        p_ip1 = p_ip1.assign_coords(lev=ds_sub["lev"])

    p_layer_top = xr.where(p_i < p_ip1, p_i, p_ip1)
    p_layer_bot = xr.where(p_i > p_ip1, p_i, p_ip1)

    p_top = p_top_hpa * 100.0
    p_bot = p_bot_hpa * 100.0

    upper = xr.where(p_layer_top > p_top, p_layer_top, p_top)
    lower = xr.where(p_layer_bot < p_bot, p_layer_bot, p_bot)
    overlap = xr.where(lower > upper, lower - upper, 0.0)

    if "lev" in ds_sub.coords:
        overlap = overlap.assign_coords(lev=ds_sub["lev"])

    overlap.name = "hybrid_overlap_dp"
    overlap.attrs.update({
        "long_name": "hybrid layer pressure-thickness overlap with target pressure range",
        "units": "Pa",
        "p_top_hpa": float(p_top_hpa),
        "p_bot_hpa": float(p_bot_hpa),
    })
    return overlap


def calc_parc_o3_hybrid(ds_sub, p_top_hpa, p_bot_hpa, verbose=False):
    """
    Partial O3 column on CAM/WACCM hybrid levels using interface-overlap dp.
    Dask-friendly version: no apply_ufunc is needed.
    """
    overlap = build_hybrid_overlap_dp(ds_sub, p_top_hpa, p_bot_hpa)

    # Preserve missing filled days.  When PS is NaN, overlap is NaN; a plain
    # xr.where(overlap > 0, ..., 0.0) would convert those days to 0 DU.
    valid_overlap = np.isfinite(overlap)
    in_target_layer = valid_overlap & (overlap > 0)
    weighted_o3 = xr.where(
        valid_overlap,
        xr.where(in_target_layer, ds_sub["O3"] * overlap * DU_FACTOR, 0.0),
        np.nan,
    )
    O3_col = weighted_o3.sum(dim="lev", skipna=False)
    O3_col.name = "O3_partial_column"
    O3_col.attrs.update({
        "long_name": f"Partial O3 column, {p_top_hpa}-{p_bot_hpa} hPa",
        "units": "DU",
        "p_top_hpa": float(p_top_hpa),
        "p_bot_hpa": float(p_bot_hpa),
        "method": "CAM/WACCM hybrid interface pressure overlap dp",
    })

    if verbose:
        dp_eff_hpa = overlap.sum("lev") / 100.0
        print(f"[Hybrid-Overlap] {p_top_hpa}-{p_bot_hpa} hPa")
        print("  dp_eff global mean (hPa):", float(dp_eff_hpa.mean().compute()))

    return O3_col


def _lat_slice_for(da, lat_bounds):
    lat_min, lat_max = lat_bounds
    lat = da["lat"]
    if float(lat[0]) <= float(lat[-1]):
        return slice(lat_min, lat_max)
    return slice(lat_max, lat_min)


def polar_cap_mean(da, lat_bounds=POLAR_LAT_BOUNDS):
    da_polar = da.sel(lat=_lat_slice_for(da, lat_bounds))
    weights = np.cos(np.deg2rad(da_polar["lat"]))
    return da_polar.weighted(weights).mean(dim=["lat", "lon"])


def _chunksizes_for(var):
    chunksizes = []
    for dim in var.dims:
        if dim in OUTPUT_CHUNKS:
            chunksizes.append(min(int(var.sizes[dim]), OUTPUT_CHUNKS[dim]))
        else:
            chunksizes.append(int(var.sizes[dim]))
    return tuple(chunksizes) if chunksizes else None


def make_encoding(ds):
    encoding = {}
    for name, var in ds.data_vars.items():
        if np.issubdtype(var.dtype, np.floating):
            item = {"zlib": True, "complevel": 1, "dtype": "float32"}
            chunksizes = _chunksizes_for(var)
            if chunksizes:
                item["chunksizes"] = chunksizes
            encoding[name] = item
        elif np.issubdtype(var.dtype, np.integer) or np.issubdtype(var.dtype, np.bool_):
            item = {"zlib": True, "complevel": 1}
            chunksizes = _chunksizes_for(var)
            if chunksizes:
                item["chunksizes"] = chunksizes
            encoding[name] = item
    return encoding


def _write_netcdf_with_fallback(ds, path, mode="w"):
    kwargs = {"mode": mode, "encoding": make_encoding(ds)}
    if NETCDF_ENGINE:
        kwargs["engine"] = NETCDF_ENGINE

    def _write_with_kwargs():
        ds.to_netcdf(path, **kwargs)

    def _write_plain():
        plain_kwargs = {"mode": mode}
        if NETCDF_ENGINE:
            plain_kwargs["engine"] = NETCDF_ENGINE
        ds.to_netcdf(path, **plain_kwargs)

    try:
        with dask.config.set(scheduler=DASK_SCHEDULER):
            if USE_PROGRESS_BAR and ProgressBar is not None:
                with ProgressBar():
                    _write_with_kwargs()
            else:
                _write_with_kwargs()
    except Exception as exc:
        print(f"  Compressed write failed for {Path(path).name}; retrying plain netCDF. Reason: {exc}")
        with dask.config.set(scheduler=DASK_SCHEDULER):
            if USE_PROGRESS_BAR and ProgressBar is not None:
                with ProgressBar():
                    _write_plain()
            else:
                _write_plain()


def save_dataset(ds, path):
    """Write via a temporary file, then replace the final path only after success."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = path.with_name(path.name + ".tmp")
    if tmp_path.exists():
        tmp_path.unlink()

    try:
        _write_netcdf_with_fallback(ds, tmp_path, mode="w")
        tmp_path.replace(path)
    except Exception:
        if tmp_path.exists():
            tmp_path.unlink()
        raise


def append_dataset_to_netcdf(ds, path):
    """Append variables to an existing aggregate NetCDF."""
    path = Path(path)
    _write_netcdf_with_fallback(ds, path, mode="a")


In [3]:
def check_hybrid_overlap_logic(sample_file=None):
    """Quick dp check before running the expensive O3 calculation."""
    if sample_file is None:
        bwcn_files = discover_o3_files(CASES["BWCN"])
        if not bwcn_files:
            raise FileNotFoundError("No BWCN O3 sample file found")
        sample_file = bwcn_files[0]

    print("Sample file:", sample_file)
    ds = xr.open_dataset(sample_file, decode_times=False).isel(time=0)
    try:
        for p_top, p_bot, tag in PRESSURE_RANGES:
            overlap = build_hybrid_overlap_dp(ds, p_top, p_bot)
            dp_eff = (overlap.sum("lev") / 100.0).load()
            expected = p_bot - p_top
            err = np.abs(dp_eff - expected)
            status = "OK" if float(err.max()) < 1.0e-6 else "CHECK"
            print(
                f"{tag}: expected {expected:.1f} hPa, "
                f"min/mean/max = {float(dp_eff.min()):.6f}/"
                f"{float(dp_eff.mean()):.6f}/{float(dp_eff.max()):.6f} hPa, "
                f"max_abs_error = {float(err.max()):.3e} [{status}]"
            )
    finally:
        ds.close()


check_hybrid_overlap_logic()


Sample file: /mnt/soclim0/public_data/weiji/BWCN/O3/BWCN.cam.h3.0001.O3.nc
30_70hPa: expected 40.0 hPa, min/mean/max = 40.000000/40.000000/40.000000 hPa, max_abs_error = 0.000e+00 [OK]
1_100hPa: expected 99.0 hPa, min/mean/max = 99.000000/99.000000/99.000000 hPa, max_abs_error = 0.000e+00 [OK]
30_100hPa: expected 70.0 hPa, min/mean/max = 70.000000/70.000000/70.000000 hPa, max_abs_error = 0.000e+00 [OK]


In [4]:
RANKING_DATA_VARS = [
    "ranked_year",
    "ranked_marapr_min_DU",
    "ranked_marapr_mean_DU",
    "ranked_is_lowest10",
    "ranked_is_lowest25pct",
]

RANKING_CSV_COLUMNS = [
    "case_name",
    "pressure_range",
    "rank_low_o3",
    "year",
    "marapr_min_DU",
    "marapr_mean_DU",
    "is_lowest10",
    "is_lowest25pct",
]


def ranking_csv_file(case_name, out_dir):
    return Path(out_dir) / f"{case_name}_partial_O3_ranking_MarApr_min_60_90N.csv"


def range_var_name(prefix, tag):
    return f"{prefix}_{tag}"


def base_case_dataset(ds, case_name, source_files):
    pressure_tags = [tag for _, _, tag in PRESSURE_RANGES]
    p_tops = [float(p_top) for p_top, _, _ in PRESSURE_RANGES]
    p_bots = [float(p_bot) for _, p_bot, _ in PRESSURE_RANGES]

    out = xr.Dataset(coords={
        "time": ds["time"],
        "lat": ds["lat"],
        "lon": ds["lon"],
        "pressure_range": ("pressure_range", pressure_tags),
        "p_top_hpa": ("pressure_range", p_tops),
        "p_bot_hpa": ("pressure_range", p_bots),
    })
    out["pressure_range"].attrs["description"] = "pressure-layer tag for partial O3 integration"
    out["p_top_hpa"].attrs["units"] = "hPa"
    out["p_bot_hpa"].attrs["units"] = "hPa"

    for aux in ["date", "datesec", "time_bnds"]:
        if aux in ds:
            out[aux] = ds[aux]

    source_files = [Path(f) for f in source_files]
    out.attrs.update({
        "title": f"Aggregate partial O3 columns for {case_name}",
        "case_name": case_name,
        "source_file_count": len(source_files),
        "source_first_file": str(source_files[0]) if source_files else "",
        "source_last_file": str(source_files[-1]) if source_files else "",
        "pressure_ranges": ", ".join(pressure_tags),
        "partial_o3_layout": "one variable per pressure range",
        "method": "O3 mol/mol integrated with exact hybrid-interface pressure overlap dp",
        "created_by": "Partial_O3_with_ranking.ipynb",
    })
    return out


def strip_coords(da, name):
    return xr.DataArray(da.data, dims=da.dims, name=name, attrs=da.attrs)


def make_one_range_dataset(ds, p_top, p_bot, tag):
    col_name = range_var_name("O3_partial_column", tag)
    ts_name = range_var_name("O3_partial_60_90N", tag)

    col = calc_parc_o3_hybrid(ds, p_top, p_bot)
    col.attrs.update({
        "long_name": f"Partial O3 column, {p_top}-{p_bot} hPa",
        "units": "DU",
        "pressure_range": tag,
        "p_top_hpa": float(p_top),
        "p_bot_hpa": float(p_bot),
        "method": "CAM/WACCM hybrid interface pressure overlap dp",
    })

    ts = polar_cap_mean(col)
    ts.attrs.update({
        "long_name": f"60-90N area-weighted mean partial O3 column, {p_top}-{p_bot} hPa",
        "units": "DU",
        "lat_bounds": "60N-90N",
        "pressure_range": tag,
        "p_top_hpa": float(p_top),
        "p_bot_hpa": float(p_bot),
    })

    return xr.Dataset({
        col_name: strip_coords(col, col_name),
        ts_name: strip_coords(ts, ts_name),
    })


def write_case_aggregate_nc(case_name, case_dir, files, out_file, overwrite=OVERWRITE):
    if out_file.exists() and not overwrite:
        if not aggregate_needs_rebuild(out_file, expected_file_count=len(files)):
            print(f"[{case_name}] Skip existing complete aggregate file: {out_file}")
            return out_file
        print(f"[{case_name}] Existing aggregate file is corrupt/incomplete; rebuilding it.")

    tmp_file = Path(out_file).with_name(Path(out_file).name + ".tmp")
    if tmp_file.exists():
        print(f"[{case_name}] Removing old temporary file: {tmp_file}")
        tmp_file.unlink()

    print(f"[{case_name}] Opening {len(files)} yearly O3 files as one dataset")
    ds = open_case_dataset(case_dir, files)
    try:
        print(f"[{case_name}] Creating aggregate file skeleton: {tmp_file}")
        base_ds = base_case_dataset(ds, case_name, files)
        save_dataset(base_ds, tmp_file)
        base_ds.close()

        for p_top, p_bot, tag in PRESSURE_RANGES:
            print(f"[{case_name}] Computing and appending {tag}")
            range_ds = make_one_range_dataset(ds, p_top, p_bot, tag)
            try:
                append_dataset_to_netcdf(range_ds, tmp_file)
            finally:
                range_ds.close()

        tmp_file.replace(out_file)
        print(f"[{case_name}] Finished aggregate file: {out_file}")
    finally:
        ds.close()
    return out_file


MONTH_ENDS_NOLEAP = np.array([31, 59, 90, 120, 151, 181, 212, 243, 273, 304, 334, 365], dtype=np.int16)


def _model_year_month_from_time_coord(time_values):
    day_index = np.asarray(time_values, dtype=float).astype(np.int64)
    years = day_index // 365 + 1
    day_of_year = day_index % 365 + 1
    months = np.searchsorted(MONTH_ENDS_NOLEAP, day_of_year, side="left") + 1
    return years.astype(np.int32), months.astype(np.int16)


def _model_year_month_from_date_or_time(date_values, time_values=None):
    date_arr = np.asarray(date_values, dtype=float)
    valid_date = np.isfinite(date_arr) & (date_arr > 0)

    years = np.full(date_arr.shape, -9999, dtype=np.int32)
    months = np.full(date_arr.shape, -1, dtype=np.int16)

    if np.any(valid_date):
        date_int = date_arr[valid_date].astype(np.int64)
        years[valid_date] = (date_int // 10000).astype(np.int32)
        months[valid_date] = ((date_int // 100) % 100).astype(np.int16)

    missing_date = ~valid_date
    if np.any(missing_date):
        if time_values is None:
            raise ValueError("CAM date has missing values and no time coordinate is available for fallback.")
        time_years, time_months = _model_year_month_from_time_coord(time_values)
        years[missing_date] = time_years[missing_date]
        months[missing_date] = time_months[missing_date]
        print(f"[INFO] Filled {int(missing_date.sum())} missing CAM date entries from no-leap time coordinate.")

    valid_time = (years > 0) & (months >= 1) & (months <= 12)
    return years, months, valid_time


def has_ranking_variables(aggregate_nc):
    if not is_valid_netcdf(aggregate_nc):
        return False
    with xr.open_dataset(aggregate_nc, decode_times=False) as ds:
        return all(name in ds.variables for name in RANKING_DATA_VARS)


def aggregate_pressure_tags(ds):
    if "pressure_range" in ds.coords:
        return list(ds["pressure_range"].values.astype(str))
    tags = []
    prefix = "O3_partial_60_90N_"
    for name in ds.data_vars:
        if name.startswith(prefix):
            tags.append(name[len(prefix):])
    return tags


def aggregate_ts_for_tag(ds, tag):
    if "O3_partial_60_90N" in ds and "pressure_range" in ds["O3_partial_60_90N"].dims:
        return ds["O3_partial_60_90N"].sel(pressure_range=tag)

    name = range_var_name("O3_partial_60_90N", tag)
    if name not in ds:
        raise KeyError(f"Cannot find polar-cap O3 time series for pressure range {tag}")
    return ds[name]


def aggregate_needs_rebuild(aggregate_nc, expected_file_count=None):
    if not is_valid_netcdf(aggregate_nc):
        return True

    with xr.open_dataset(aggregate_nc, decode_times=False) as ds:
        n_time = int(ds.sizes.get("time", 0))
        if expected_file_count is not None:
            if int(ds.attrs.get("source_file_count", -1)) != int(expected_file_count):
                print(f"  Aggregate source_file_count does not match current file list: {aggregate_nc}")
                return True

        if "date" in ds:
            date_values = np.asarray(ds["date"].values, dtype=float)
            bad_date_count = int((~np.isfinite(date_values) | (date_values <= 0)).sum())
            if bad_date_count:
                print(f"  Aggregate has {bad_date_count} missing/invalid CAM date entries: {aggregate_nc}")
                return True

        for tag in aggregate_pressure_tags(ds):
            try:
                ts = np.asarray(aggregate_ts_for_tag(ds, tag).values, dtype=float)
            except KeyError:
                continue
            if ts.ndim != 1 or n_time < 365:
                continue
            finite_ts = ts[np.isfinite(ts)]
            zero_day_count = int((np.abs(finite_ts) <= 1.0e-12).sum())
            if zero_day_count:
                print(f"  Aggregate has {zero_day_count} zero-filled {tag} days and will be rebuilt: {aggregate_nc}")
                return True
            zero_years = []
            for year_idx in range(n_time // 365):
                year_vals = ts[year_idx * 365:(year_idx + 1) * 365]
                finite_vals = year_vals[np.isfinite(year_vals)]
                if finite_vals.size and np.nanmax(np.abs(finite_vals)) == 0.0:
                    zero_years.append(year_idx + 1)
            if zero_years:
                print(f"  Aggregate has zero-filled {tag} years, first examples: {zero_years[:5]}")
                return True

    return False


def make_ranking_dataset_from_aggregate(case_name, aggregate_nc):
    # decode_times=False is intentional: CAM no-leap time/date are handled explicitly.
    ds = xr.open_dataset(aggregate_nc, chunks={"time": OUTPUT_CHUNKS["time"]}, decode_times=False)
    try:
        time_vals = ds["time"].values if "time" in ds.coords else None
        if "date" in ds:
            years_by_time, months_by_time, valid_time_mask = _model_year_month_from_date_or_time(ds["date"].values, time_vals)
        elif time_vals is not None:
            years_by_time, months_by_time = _model_year_month_from_time_coord(time_vals)
            valid_time_mask = np.ones(years_by_time.shape, dtype=bool)
            print("[INFO] Aggregate has no CAM date variable; using no-leap time coordinate for ranking.")
        else:
            raise KeyError("Aggregate file has neither CAM date nor time coordinate; cannot build model-year ranking.")

        unique_years = np.array(sorted(np.unique(years_by_time[valid_time_mask])), dtype=np.int32)
        pressure_tags = aggregate_pressure_tags(ds)

        n_pressure = len(pressure_tags)
        max_rank = len(unique_years)

        ranked_year = np.full((n_pressure, max_rank), -9999, dtype=np.int32)
        ranked_min = np.full((n_pressure, max_rank), np.nan, dtype=np.float32)
        ranked_mean = np.full((n_pressure, max_rank), np.nan, dtype=np.float32)
        is_lowest10 = np.zeros((n_pressure, max_rank), dtype=np.int8)
        is_lowest25pct = np.zeros((n_pressure, max_rank), dtype=np.int8)

        spring_month_mask = valid_time_mask & np.isin(months_by_time, RANKING_MONTHS)

        for ip, tag in enumerate(pressure_tags):
            ts = np.asarray(aggregate_ts_for_tag(ds, tag).load().values, dtype=float)
            records = []
            for year in unique_years:
                mask = (years_by_time == year) & spring_month_mask
                if not np.any(mask):
                    continue
                vals = ts[mask]
                finite_vals = vals[np.isfinite(vals)]
                valid_vals = finite_vals[finite_vals > RANKING_MIN_VALID_DU]
                if valid_vals.size < RANKING_MIN_VALID_DAYS:
                    print(
                        f"[{case_name}] Skip year {int(year)} for {tag}: "
                        f"only {valid_vals.size} finite March-April O3 days above {RANKING_MIN_VALID_DU:g} DU "
                        f"(< {RANKING_MIN_VALID_DAYS})"
                    )
                    continue
                records.append((int(year), float(np.min(valid_vals)), float(np.mean(valid_vals))))

            records = sorted(records, key=lambda item: item[1])
            n_low10 = min(10, len(records))
            n_low25 = max(int(0.25 * len(records)), 1) if records else 0

            for rank_idx, (year, min_val, mean_val) in enumerate(records):
                ranked_year[ip, rank_idx] = year
                ranked_min[ip, rank_idx] = min_val
                ranked_mean[ip, rank_idx] = mean_val
                is_lowest10[ip, rank_idx] = int(rank_idx < n_low10)
                is_lowest25pct[ip, rank_idx] = int(rank_idx < n_low25)

            print(f"[{case_name}] {tag} lowest 10 event years: {ranked_year[ip, :n_low10]}")

        rank_coord = np.arange(1, max_rank + 1, dtype=np.int32)
        ranking_ds = xr.Dataset(
            {
                "ranked_year": (("pressure_range", "rank_low_o3"), ranked_year),
                "ranked_marapr_min_DU": (("pressure_range", "rank_low_o3"), ranked_min),
                "ranked_marapr_mean_DU": (("pressure_range", "rank_low_o3"), ranked_mean),
                "ranked_is_lowest10": (("pressure_range", "rank_low_o3"), is_lowest10),
                "ranked_is_lowest25pct": (("pressure_range", "rank_low_o3"), is_lowest25pct),
            },
            coords={"pressure_range": pressure_tags, "rank_low_o3": rank_coord},
        )
        ranking_ds["rank_low_o3"].attrs.update({
            "long_name": "rank after sorting March-April minimum 60-90N partial O3 from low to high",
        })
        ranking_ds["ranked_year"].attrs.update({
            "long_name": "model event year sorted by low March-April partial O3",
            "missing_value": np.int32(-9999),
        })
        ranking_ds["ranked_marapr_min_DU"].attrs.update({
            "long_name": "March-April minimum of 60-90N partial O3 for ranked model year",
            "units": "DU",
        })
        ranking_ds["ranked_marapr_mean_DU"].attrs.update({
            "long_name": "March-April mean of 60-90N partial O3 for ranked model year",
            "units": "DU",
        })
        ranking_ds["ranked_is_lowest10"].attrs.update({
            "long_name": "1 if rank is in lowest 10 model years, else 0",
        })
        ranking_ds["ranked_is_lowest25pct"].attrs.update({
            "long_name": "1 if rank is in lowest 25 percent model years, else 0",
        })
        ranking_ds.attrs.update({
            "ranking_source": "CAM date variable (YYYYMMDD), with no-leap time fallback for missing date entries",
            "ranking_months": ",".join(str(m) for m in RANKING_MONTHS),
        })
        return ranking_ds
    finally:
        ds.close()


def ranking_rows_from_dataset(case_name, ranking_ds):
    pressure_tags = ranking_ds["pressure_range"].values.astype(str)
    ranks = ranking_ds["rank_low_o3"].values.astype(int)
    years = np.asarray(ranking_ds["ranked_year"].values, dtype=float)
    mins = np.asarray(ranking_ds["ranked_marapr_min_DU"].values, dtype=float)
    means = np.asarray(ranking_ds["ranked_marapr_mean_DU"].values, dtype=float)
    low10 = np.asarray(ranking_ds["ranked_is_lowest10"].values, dtype=float)
    low25 = np.asarray(ranking_ds["ranked_is_lowest25pct"].values, dtype=float)

    rows = []
    for ip, tag in enumerate(pressure_tags):
        for ir, rank in enumerate(ranks):
            year_value = years[ip, ir]
            if (not np.isfinite(year_value)) or int(year_value) == -9999:
                continue
            rows.append({
                "case_name": case_name,
                "pressure_range": tag,
                "rank_low_o3": int(rank),
                "year": int(year_value),
                "marapr_min_DU": float(mins[ip, ir]),
                "marapr_mean_DU": float(means[ip, ir]),
                "is_lowest10": bool(low10[ip, ir]) if np.isfinite(low10[ip, ir]) else False,
                "is_lowest25pct": bool(low25[ip, ir]) if np.isfinite(low25[ip, ir]) else False,
            })
    return rows


def write_ranking_csv(case_name, out_dir, ranking_ds):
    csv_path = ranking_csv_file(case_name, out_dir)
    rows = ranking_rows_from_dataset(case_name, ranking_ds)
    with csv_path.open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=RANKING_CSV_COLUMNS)
        writer.writeheader()
        writer.writerows(rows)
    print(f"[{case_name}] Saved ranking CSV: {csv_path}")
    return csv_path


def write_ranking_csv_from_aggregate(case_name, out_dir, aggregate_nc):
    # Recompute from the aggregate daily O3 time series so CSV output follows the
    # current validity rules even when older ranking variables already exist in NetCDF.
    ranking_ds = make_ranking_dataset_from_aggregate(case_name, aggregate_nc)
    try:
        return write_ranking_csv(case_name, out_dir, ranking_ds)
    finally:
        ranking_ds.close()


def append_ranking_to_aggregate_nc(case_name, aggregate_nc, overwrite=OVERWRITE):
    out_dir = Path(aggregate_nc).parent

    if has_ranking_variables(aggregate_nc) and not overwrite:
        print(f"[{case_name}] Ranking variables already exist in {aggregate_nc}; skip NetCDF append.")
        print(f"[{case_name}] Recomputing ranking CSV from daily aggregate data with current validity rules.")
        write_ranking_csv_from_aggregate(case_name, out_dir, aggregate_nc)
        return aggregate_nc

    print(f"[{case_name}] Building ranking variables from aggregate NetCDF")
    ranking_ds = make_ranking_dataset_from_aggregate(case_name, aggregate_nc)
    try:
        print(f"[{case_name}] Appending ranking variables into: {aggregate_nc}")
        append_ds = ranking_ds.drop_vars("pressure_range") if "pressure_range" in ranking_ds.coords else ranking_ds
        append_dataset_to_netcdf(append_ds, aggregate_nc)
        write_ranking_csv(case_name, out_dir, ranking_ds)
    finally:
        ranking_ds.close()
    return aggregate_nc



# -----------------------------------------------------------------------------
# Updated ranking implementation: 5-day running mean + low/high ozone years.
# -----------------------------------------------------------------------------
RANKING_DATA_VARS = [
    "ranked_year",
    "ranked_marapr_min_DU",
    "ranked_marapr_max_DU",
    "ranked_marapr_mean_DU",
    "ranked_is_lowest10",
    "ranked_is_lowest25pct",
    "ranked_high_year",
    "ranked_high_marapr_min_DU",
    "ranked_high_marapr_max_DU",
    "ranked_high_marapr_mean_DU",
    "ranked_is_highest10",
    "ranked_is_highest25pct",
]

RANKING_CSV_COLUMNS = [
    "case_name",
    "pressure_range",
    "rank_low_o3",
    "rank_high_o3",
    "year",
    "marapr_min_DU",
    "marapr_max_DU",
    "marapr_mean_DU",
    "ranking_rolling_days",
    "ranking_rolling_min_periods",
    "is_lowest10",
    "is_lowest25pct",
    "is_highest10",
    "is_highest25pct",
]


def aggregate_has_any_ranking_variables(aggregate_nc):
    if not is_valid_netcdf(aggregate_nc):
        return False
    with xr.open_dataset(aggregate_nc, decode_times=False) as ds:
        ranking_like = [name for name in ds.variables if name.startswith("ranked_") or name.startswith("rank_high_o3")]
        return bool(ranking_like)


def _ranking_series_for_extremes(ts):
    da = xr.DataArray(np.asarray(ts, dtype=float), dims=("time",))
    da = da.where(da > RANKING_MIN_VALID_DU)
    if RANKING_ROLLING_DAYS and RANKING_ROLLING_DAYS > 1:
        da = da.rolling(
            time=RANKING_ROLLING_DAYS,
            center=RANKING_ROLLING_CENTER,
            min_periods=RANKING_ROLLING_MIN_PERIODS,
        ).mean()
    return np.asarray(da.values, dtype=float)


def _records_for_pressure_tag(ts_rank, years_by_time, spring_month_mask, unique_years):
    records = []
    for year in unique_years:
        mask = (years_by_time == year) & spring_month_mask
        if not np.any(mask):
            continue
        vals = ts_rank[mask]
        valid_vals = vals[np.isfinite(vals) & (vals > RANKING_MIN_VALID_DU)]
        if valid_vals.size < RANKING_MIN_VALID_DAYS:
            continue
        records.append({
            "year": int(year),
            "marapr_min_DU": float(np.nanmin(valid_vals)),
            "marapr_max_DU": float(np.nanmax(valid_vals)),
            "marapr_mean_DU": float(np.nanmean(valid_vals)),
        })
    return records


def make_ranking_dataset_from_aggregate(case_name, aggregate_nc):
    # decode_times=False is intentional: CAM no-leap time/date are handled explicitly.
    ds = xr.open_dataset(aggregate_nc, chunks={"time": OUTPUT_CHUNKS["time"]}, decode_times=False)
    try:
        time_vals = ds["time"].values if "time" in ds.coords else None
        if "date" in ds:
            years_by_time, months_by_time, valid_time_mask = _model_year_month_from_date_or_time(ds["date"].values, time_vals)
        elif time_vals is not None:
            years_by_time, months_by_time = _model_year_month_from_time_coord(time_vals)
            valid_time_mask = np.ones(years_by_time.shape, dtype=bool)
            print("[INFO] Aggregate has no CAM date variable; using no-leap time coordinate for ranking.")
        else:
            raise KeyError("Aggregate file has neither CAM date nor time coordinate; cannot build model-year ranking.")

        unique_years = np.array(sorted(np.unique(years_by_time[valid_time_mask])), dtype=np.int32)
        pressure_tags = aggregate_pressure_tags(ds)
        n_pressure = len(pressure_tags)
        max_rank = len(unique_years)

        ranked_year = np.full((n_pressure, max_rank), -9999, dtype=np.int32)
        ranked_min = np.full((n_pressure, max_rank), np.nan, dtype=np.float32)
        ranked_max = np.full((n_pressure, max_rank), np.nan, dtype=np.float32)
        ranked_mean = np.full((n_pressure, max_rank), np.nan, dtype=np.float32)
        is_lowest10 = np.zeros((n_pressure, max_rank), dtype=np.int8)
        is_lowest25pct = np.zeros((n_pressure, max_rank), dtype=np.int8)

        ranked_high_year = np.full((n_pressure, max_rank), -9999, dtype=np.int32)
        ranked_high_min = np.full((n_pressure, max_rank), np.nan, dtype=np.float32)
        ranked_high_max = np.full((n_pressure, max_rank), np.nan, dtype=np.float32)
        ranked_high_mean = np.full((n_pressure, max_rank), np.nan, dtype=np.float32)
        is_highest10 = np.zeros((n_pressure, max_rank), dtype=np.int8)
        is_highest25pct = np.zeros((n_pressure, max_rank), dtype=np.int8)

        spring_month_mask = valid_time_mask & np.isin(months_by_time, RANKING_MONTHS)

        for ip, tag in enumerate(pressure_tags):
            ts_raw = np.asarray(aggregate_ts_for_tag(ds, tag).load().values, dtype=float)
            ts_rank = _ranking_series_for_extremes(ts_raw)
            records = _records_for_pressure_tag(ts_rank, years_by_time, spring_month_mask, unique_years)

            skipped = len(unique_years) - len(records)
            if skipped:
                print(f"[{case_name}] {tag}: skipped {skipped} years with insufficient valid 5-day-smoothed O3 days")

            low_records = sorted(records, key=lambda item: item["marapr_min_DU"])
            high_records = sorted(records, key=lambda item: item["marapr_max_DU"], reverse=True)
            n_low10 = min(10, len(low_records))
            n_high10 = min(10, len(high_records))
            n_low25 = max(int(0.25 * len(low_records)), 1) if low_records else 0
            n_high25 = max(int(0.25 * len(high_records)), 1) if high_records else 0

            for rank_idx, rec in enumerate(low_records):
                ranked_year[ip, rank_idx] = rec["year"]
                ranked_min[ip, rank_idx] = rec["marapr_min_DU"]
                ranked_max[ip, rank_idx] = rec["marapr_max_DU"]
                ranked_mean[ip, rank_idx] = rec["marapr_mean_DU"]
                is_lowest10[ip, rank_idx] = int(rank_idx < n_low10)
                is_lowest25pct[ip, rank_idx] = int(rank_idx < n_low25)

            for rank_idx, rec in enumerate(high_records):
                ranked_high_year[ip, rank_idx] = rec["year"]
                ranked_high_min[ip, rank_idx] = rec["marapr_min_DU"]
                ranked_high_max[ip, rank_idx] = rec["marapr_max_DU"]
                ranked_high_mean[ip, rank_idx] = rec["marapr_mean_DU"]
                is_highest10[ip, rank_idx] = int(rank_idx < n_high10)
                is_highest25pct[ip, rank_idx] = int(rank_idx < n_high25)

            print(f"[{case_name}] {tag} lowest 10 event years: {ranked_year[ip, :n_low10]}")
            print(f"[{case_name}] {tag} highest 10 event years: {ranked_high_year[ip, :n_high10]}")

        low_rank_coord = np.arange(1, max_rank + 1, dtype=np.int32)
        high_rank_coord = np.arange(1, max_rank + 1, dtype=np.int32)
        ranking_ds = xr.Dataset(
            {
                "ranked_year": (("pressure_range", "rank_low_o3"), ranked_year),
                "ranked_marapr_min_DU": (("pressure_range", "rank_low_o3"), ranked_min),
                "ranked_marapr_max_DU": (("pressure_range", "rank_low_o3"), ranked_max),
                "ranked_marapr_mean_DU": (("pressure_range", "rank_low_o3"), ranked_mean),
                "ranked_is_lowest10": (("pressure_range", "rank_low_o3"), is_lowest10),
                "ranked_is_lowest25pct": (("pressure_range", "rank_low_o3"), is_lowest25pct),
                "ranked_high_year": (("pressure_range", "rank_high_o3"), ranked_high_year),
                "ranked_high_marapr_min_DU": (("pressure_range", "rank_high_o3"), ranked_high_min),
                "ranked_high_marapr_max_DU": (("pressure_range", "rank_high_o3"), ranked_high_max),
                "ranked_high_marapr_mean_DU": (("pressure_range", "rank_high_o3"), ranked_high_mean),
                "ranked_is_highest10": (("pressure_range", "rank_high_o3"), is_highest10),
                "ranked_is_highest25pct": (("pressure_range", "rank_high_o3"), is_highest25pct),
            },
            coords={"pressure_range": pressure_tags, "rank_low_o3": low_rank_coord, "rank_high_o3": high_rank_coord},
        )
        ranking_ds["rank_low_o3"].attrs.update({"long_name": "rank after sorting 5-day-smoothed March-April minimum 60-90N partial O3 from low to high"})
        ranking_ds["rank_high_o3"].attrs.update({"long_name": "rank after sorting 5-day-smoothed March-April maximum 60-90N partial O3 from high to low"})
        ranking_ds["ranked_year"].attrs.update({"long_name": "model event year sorted by low March-April partial O3", "missing_value": np.int32(-9999)})
        ranking_ds["ranked_high_year"].attrs.update({"long_name": "model event year sorted by high March-April partial O3", "missing_value": np.int32(-9999)})
        for name in ["ranked_marapr_min_DU", "ranked_high_marapr_min_DU"]:
            ranking_ds[name].attrs.update({"long_name": "March-April minimum of 5-day-smoothed 60-90N partial O3 for ranked model year", "units": "DU"})
        for name in ["ranked_marapr_max_DU", "ranked_high_marapr_max_DU"]:
            ranking_ds[name].attrs.update({"long_name": "March-April maximum of 5-day-smoothed 60-90N partial O3 for ranked model year", "units": "DU"})
        for name in ["ranked_marapr_mean_DU", "ranked_high_marapr_mean_DU"]:
            ranking_ds[name].attrs.update({"long_name": "March-April mean of 5-day-smoothed 60-90N partial O3 for ranked model year", "units": "DU"})
        ranking_ds["ranked_is_lowest10"].attrs.update({"long_name": "1 if rank is in lowest 10 model years, else 0"})
        ranking_ds["ranked_is_lowest25pct"].attrs.update({"long_name": "1 if rank is in lowest 25 percent model years, else 0"})
        ranking_ds["ranked_is_highest10"].attrs.update({"long_name": "1 if rank is in highest 10 model years, else 0"})
        ranking_ds["ranked_is_highest25pct"].attrs.update({"long_name": "1 if rank is in highest 25 percent model years, else 0"})
        ranking_ds.attrs.update({
            "ranking_source": "CAM date variable (YYYYMMDD), with no-leap time fallback for missing date entries",
            "ranking_months": ",".join(str(m) for m in RANKING_MONTHS),
            "ranking_rolling_days": int(RANKING_ROLLING_DAYS),
            "ranking_rolling_center": str(RANKING_ROLLING_CENTER),
            "ranking_rolling_min_periods": int(RANKING_ROLLING_MIN_PERIODS),
            "ranking_valid_min_DU": float(RANKING_MIN_VALID_DU),
        })
        return ranking_ds
    finally:
        ds.close()


def _flag_array(ranking_ds, name):
    return np.asarray(ranking_ds[name].values, dtype=float)


def ranking_rows_from_dataset(case_name, ranking_ds):
    pressure_tags = ranking_ds["pressure_range"].values.astype(str)
    low_ranks = ranking_ds["rank_low_o3"].values.astype(int)
    high_ranks = ranking_ds["rank_high_o3"].values.astype(int)

    low_years = np.asarray(ranking_ds["ranked_year"].values, dtype=float)
    low_mins = np.asarray(ranking_ds["ranked_marapr_min_DU"].values, dtype=float)
    low_maxs = np.asarray(ranking_ds["ranked_marapr_max_DU"].values, dtype=float)
    low_means = np.asarray(ranking_ds["ranked_marapr_mean_DU"].values, dtype=float)
    low10 = _flag_array(ranking_ds, "ranked_is_lowest10")
    low25 = _flag_array(ranking_ds, "ranked_is_lowest25pct")

    high_years = np.asarray(ranking_ds["ranked_high_year"].values, dtype=float)
    high_mins = np.asarray(ranking_ds["ranked_high_marapr_min_DU"].values, dtype=float)
    high_maxs = np.asarray(ranking_ds["ranked_high_marapr_max_DU"].values, dtype=float)
    high_means = np.asarray(ranking_ds["ranked_high_marapr_mean_DU"].values, dtype=float)
    high10 = _flag_array(ranking_ds, "ranked_is_highest10")
    high25 = _flag_array(ranking_ds, "ranked_is_highest25pct")

    rows = []
    for ip, tag in enumerate(pressure_tags):
        high_by_year = {}
        for ir, rank in enumerate(high_ranks):
            year_value = high_years[ip, ir]
            if (not np.isfinite(year_value)) or int(year_value) == -9999:
                continue
            high_by_year[int(year_value)] = {
                "rank_high_o3": int(rank),
                "is_highest10": bool(high10[ip, ir]) if np.isfinite(high10[ip, ir]) else False,
                "is_highest25pct": bool(high25[ip, ir]) if np.isfinite(high25[ip, ir]) else False,
                "high_min": float(high_mins[ip, ir]),
                "high_max": float(high_maxs[ip, ir]),
                "high_mean": float(high_means[ip, ir]),
            }

        for ir, rank in enumerate(low_ranks):
            year_value = low_years[ip, ir]
            if (not np.isfinite(year_value)) or int(year_value) == -9999:
                continue
            year = int(year_value)
            high_rec = high_by_year.get(year, {})
            rows.append({
                "case_name": case_name,
                "pressure_range": tag,
                "rank_low_o3": int(rank),
                "rank_high_o3": high_rec.get("rank_high_o3", -9999),
                "year": year,
                "marapr_min_DU": float(low_mins[ip, ir]),
                "marapr_max_DU": float(low_maxs[ip, ir]),
                "marapr_mean_DU": float(low_means[ip, ir]),
                "ranking_rolling_days": int(RANKING_ROLLING_DAYS),
                "ranking_rolling_min_periods": int(RANKING_ROLLING_MIN_PERIODS),
                "is_lowest10": bool(low10[ip, ir]) if np.isfinite(low10[ip, ir]) else False,
                "is_lowest25pct": bool(low25[ip, ir]) if np.isfinite(low25[ip, ir]) else False,
                "is_highest10": bool(high_rec.get("is_highest10", False)),
                "is_highest25pct": bool(high_rec.get("is_highest25pct", False)),
            })
    return rows


def write_ranking_csv(case_name, out_dir, ranking_ds):
    csv_path = ranking_csv_file(case_name, out_dir)
    rows = ranking_rows_from_dataset(case_name, ranking_ds)
    with csv_path.open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=RANKING_CSV_COLUMNS)
        writer.writeheader()
        writer.writerows(rows)
    print(f"[{case_name}] Saved 5-day-smoothed low/high ranking CSV: {csv_path}")
    return csv_path


def write_ranking_csv_from_aggregate(case_name, out_dir, aggregate_nc):
    ranking_ds = make_ranking_dataset_from_aggregate(case_name, aggregate_nc)
    try:
        return write_ranking_csv(case_name, out_dir, ranking_ds)
    finally:
        ranking_ds.close()


def append_ranking_to_aggregate_nc(case_name, aggregate_nc, overwrite=OVERWRITE):
    out_dir = Path(aggregate_nc).parent

    if RANKING_CSV_ONLY:
        print(f"[{case_name}] RANKING_CSV_ONLY=True: rebuilding CSV only; NetCDF ranking variables unchanged.")
        write_ranking_csv_from_aggregate(case_name, out_dir, aggregate_nc)
        return aggregate_nc

    if aggregate_has_any_ranking_variables(aggregate_nc) and not overwrite:
        print(f"[{case_name}] Ranking-like variables already exist in {aggregate_nc}; skip NetCDF append.")
        print(f"[{case_name}] Recomputing 5-day-smoothed low/high ranking CSV from daily aggregate data.")
        write_ranking_csv_from_aggregate(case_name, out_dir, aggregate_nc)
        return aggregate_nc

    print(f"[{case_name}] Building 5-day-smoothed low/high ranking variables from aggregate NetCDF")
    ranking_ds = make_ranking_dataset_from_aggregate(case_name, aggregate_nc)
    try:
        print(f"[{case_name}] Appending ranking variables into: {aggregate_nc}")
        append_ds = ranking_ds.drop_vars("pressure_range") if "pressure_range" in ranking_ds.coords else ranking_ds
        append_dataset_to_netcdf(append_ds, aggregate_nc)
        write_ranking_csv(case_name, out_dir, ranking_ds)
    finally:
        ranking_ds.close()
    return aggregate_nc



# -----------------------------------------------------------------------------
# Final ranking implementation: raw + 5-day-running-mean, low25 + high25.
# CSV output is one row per pressure_range/year with explicit *_raw and *_rm5 flags.
# -----------------------------------------------------------------------------
RANKING_METHODS = {
    "raw": {"rolling_days": 1, "center": False, "min_periods": 1},
    "rm5": {"rolling_days": 5, "center": True, "min_periods": 5},
}
RANKING_METHOD_ORDER = ["raw", "rm5"]

RANKING_DATA_VARS = [
    "ranked_low_year",
    "ranked_low_marapr_min_DU",
    "ranked_low_marapr_max_DU",
    "ranked_low_marapr_mean_DU",
    "ranked_low_is_lowest10",
    "ranked_low_is_lowest25pct",
    "ranked_high_year",
    "ranked_high_marapr_min_DU",
    "ranked_high_marapr_max_DU",
    "ranked_high_marapr_mean_DU",
    "ranked_high_is_highest10",
    "ranked_high_is_highest25pct",
]

RANKING_CSV_COLUMNS = ["case_name", "pressure_range", "year"]
for _method in RANKING_METHOD_ORDER:
    RANKING_CSV_COLUMNS.extend([
        f"ranking_rolling_days_{_method}",
        f"ranking_rolling_min_periods_{_method}",
        f"marapr_min_DU_{_method}",
        f"marapr_max_DU_{_method}",
        f"marapr_mean_DU_{_method}",
        f"rank_low_o3_{_method}",
        f"rank_high_o3_{_method}",
        f"is_lowest10_{_method}",
        f"is_lowest25pct_{_method}",
        f"is_highest10_{_method}",
        f"is_highest25pct_{_method}",
    ])


def has_ranking_variables(aggregate_nc):
    if not is_valid_netcdf(aggregate_nc):
        return False
    with xr.open_dataset(aggregate_nc, decode_times=False) as ds:
        return all(name in ds.variables for name in RANKING_DATA_VARS)


def aggregate_has_any_ranking_variables(aggregate_nc):
    if not is_valid_netcdf(aggregate_nc):
        return False
    with xr.open_dataset(aggregate_nc, decode_times=False) as ds:
        ranking_like = [name for name in ds.variables if name.startswith("ranked_") or name.startswith("rank_low_o3") or name.startswith("rank_high_o3")]
        return bool(ranking_like)


def _ranking_method_series(ts, method):
    spec = RANKING_METHODS[method]
    da = xr.DataArray(np.asarray(ts, dtype=float), dims=("time",))
    da = da.where(da > RANKING_MIN_VALID_DU)
    if spec["rolling_days"] and spec["rolling_days"] > 1:
        da = da.rolling(
            time=spec["rolling_days"],
            center=spec["center"],
            min_periods=spec["min_periods"],
        ).mean()
    return np.asarray(da.values, dtype=float)


def _records_for_pressure_tag(ts_rank, years_by_time, spring_month_mask, unique_years):
    records = []
    for year in unique_years:
        mask = (years_by_time == year) & spring_month_mask
        if not np.any(mask):
            continue
        vals = ts_rank[mask]
        valid_vals = vals[np.isfinite(vals) & (vals > RANKING_MIN_VALID_DU)]
        if valid_vals.size < RANKING_MIN_VALID_DAYS:
            continue
        records.append({
            "year": int(year),
            "marapr_min_DU": float(np.nanmin(valid_vals)),
            "marapr_max_DU": float(np.nanmax(valid_vals)),
            "marapr_mean_DU": float(np.nanmean(valid_vals)),
        })
    return records


def _rank_records(records):
    low_records = sorted(records, key=lambda item: item["marapr_min_DU"])
    high_records = sorted(records, key=lambda item: item["marapr_max_DU"], reverse=True)
    n_low10 = min(10, len(low_records))
    n_high10 = min(10, len(high_records))
    n_low25 = max(int(0.25 * len(low_records)), 1) if low_records else 0
    n_high25 = max(int(0.25 * len(high_records)), 1) if high_records else 0

    by_year = {}
    for rank_idx, rec in enumerate(low_records, start=1):
        info = by_year.setdefault(rec["year"], dict(rec))
        info.update({
            "rank_low_o3": rank_idx,
            "is_lowest10": rank_idx <= n_low10,
            "is_lowest25pct": rank_idx <= n_low25,
        })
    for rank_idx, rec in enumerate(high_records, start=1):
        info = by_year.setdefault(rec["year"], dict(rec))
        info.update({
            "rank_high_o3": rank_idx,
            "is_highest10": rank_idx <= n_high10,
            "is_highest25pct": rank_idx <= n_high25,
        })
    return low_records, high_records, by_year


def make_ranking_dataset_from_aggregate(case_name, aggregate_nc):
    ds = xr.open_dataset(aggregate_nc, chunks={"time": OUTPUT_CHUNKS["time"]}, decode_times=False)
    try:
        time_vals = ds["time"].values if "time" in ds.coords else None
        if "date" in ds:
            years_by_time, months_by_time, valid_time_mask = _model_year_month_from_date_or_time(ds["date"].values, time_vals)
        elif time_vals is not None:
            years_by_time, months_by_time = _model_year_month_from_time_coord(time_vals)
            valid_time_mask = np.ones(years_by_time.shape, dtype=bool)
            print("[INFO] Aggregate has no CAM date variable; using no-leap time coordinate for ranking.")
        else:
            raise KeyError("Aggregate file has neither CAM date nor time coordinate; cannot build model-year ranking.")

        unique_years = np.array(sorted(np.unique(years_by_time[valid_time_mask])), dtype=np.int32)
        pressure_tags = aggregate_pressure_tags(ds)
        method_names = np.asarray(RANKING_METHOD_ORDER, dtype=object)
        n_pressure = len(pressure_tags)
        n_method = len(method_names)
        max_rank = len(unique_years)

        low_year = np.full((n_pressure, n_method, max_rank), -9999, dtype=np.int32)
        low_min = np.full((n_pressure, n_method, max_rank), np.nan, dtype=np.float32)
        low_max = np.full((n_pressure, n_method, max_rank), np.nan, dtype=np.float32)
        low_mean = np.full((n_pressure, n_method, max_rank), np.nan, dtype=np.float32)
        low10 = np.zeros((n_pressure, n_method, max_rank), dtype=np.int8)
        low25 = np.zeros((n_pressure, n_method, max_rank), dtype=np.int8)

        high_year = np.full((n_pressure, n_method, max_rank), -9999, dtype=np.int32)
        high_min = np.full((n_pressure, n_method, max_rank), np.nan, dtype=np.float32)
        high_max = np.full((n_pressure, n_method, max_rank), np.nan, dtype=np.float32)
        high_mean = np.full((n_pressure, n_method, max_rank), np.nan, dtype=np.float32)
        high10 = np.zeros((n_pressure, n_method, max_rank), dtype=np.int8)
        high25 = np.zeros((n_pressure, n_method, max_rank), dtype=np.int8)

        csv_records = {}
        spring_month_mask = valid_time_mask & np.isin(months_by_time, RANKING_MONTHS)

        for ip, tag in enumerate(pressure_tags):
            ts_raw = np.asarray(aggregate_ts_for_tag(ds, tag).load().values, dtype=float)
            for im, method in enumerate(RANKING_METHOD_ORDER):
                ts_rank = _ranking_method_series(ts_raw, method)
                records = _records_for_pressure_tag(ts_rank, years_by_time, spring_month_mask, unique_years)
                skipped = len(unique_years) - len(records)
                if skipped:
                    print(f"[{case_name}] {tag} {method}: skipped {skipped} years with insufficient valid O3 days")
                low_records, high_records, by_year = _rank_records(records)

                for rank_idx, rec in enumerate(low_records):
                    low_year[ip, im, rank_idx] = rec["year"]
                    low_min[ip, im, rank_idx] = rec["marapr_min_DU"]
                    low_max[ip, im, rank_idx] = rec["marapr_max_DU"]
                    low_mean[ip, im, rank_idx] = rec["marapr_mean_DU"]
                    low10[ip, im, rank_idx] = int(rank_idx < min(10, len(low_records)))
                    low25[ip, im, rank_idx] = int(rank_idx < (max(int(0.25 * len(low_records)), 1) if low_records else 0))

                for rank_idx, rec in enumerate(high_records):
                    high_year[ip, im, rank_idx] = rec["year"]
                    high_min[ip, im, rank_idx] = rec["marapr_min_DU"]
                    high_max[ip, im, rank_idx] = rec["marapr_max_DU"]
                    high_mean[ip, im, rank_idx] = rec["marapr_mean_DU"]
                    high10[ip, im, rank_idx] = int(rank_idx < min(10, len(high_records)))
                    high25[ip, im, rank_idx] = int(rank_idx < (max(int(0.25 * len(high_records)), 1) if high_records else 0))

                for year, info in by_year.items():
                    row = csv_records.setdefault((tag, year), {"case_name": case_name, "pressure_range": tag, "year": year})
                    spec = RANKING_METHODS[method]
                    row.update({
                        f"ranking_rolling_days_{method}": int(spec["rolling_days"]),
                        f"ranking_rolling_min_periods_{method}": int(spec["min_periods"]),
                        f"marapr_min_DU_{method}": info.get("marapr_min_DU", np.nan),
                        f"marapr_max_DU_{method}": info.get("marapr_max_DU", np.nan),
                        f"marapr_mean_DU_{method}": info.get("marapr_mean_DU", np.nan),
                        f"rank_low_o3_{method}": int(info.get("rank_low_o3", -9999)),
                        f"rank_high_o3_{method}": int(info.get("rank_high_o3", -9999)),
                        f"is_lowest10_{method}": bool(info.get("is_lowest10", False)),
                        f"is_lowest25pct_{method}": bool(info.get("is_lowest25pct", False)),
                        f"is_highest10_{method}": bool(info.get("is_highest10", False)),
                        f"is_highest25pct_{method}": bool(info.get("is_highest25pct", False)),
                    })

                print(f"[{case_name}] {tag} {method} lowest 10 event years: {low_year[ip, im, :min(10, len(low_records))]}")
                print(f"[{case_name}] {tag} {method} highest 10 event years: {high_year[ip, im, :min(10, len(high_records))]}")

        rank_coord = np.arange(1, max_rank + 1, dtype=np.int32)
        ranking_ds = xr.Dataset(
            {
                "ranked_low_year": (("pressure_range", "ranking_method", "rank_o3"), low_year),
                "ranked_low_marapr_min_DU": (("pressure_range", "ranking_method", "rank_o3"), low_min),
                "ranked_low_marapr_max_DU": (("pressure_range", "ranking_method", "rank_o3"), low_max),
                "ranked_low_marapr_mean_DU": (("pressure_range", "ranking_method", "rank_o3"), low_mean),
                "ranked_low_is_lowest10": (("pressure_range", "ranking_method", "rank_o3"), low10),
                "ranked_low_is_lowest25pct": (("pressure_range", "ranking_method", "rank_o3"), low25),
                "ranked_high_year": (("pressure_range", "ranking_method", "rank_o3"), high_year),
                "ranked_high_marapr_min_DU": (("pressure_range", "ranking_method", "rank_o3"), high_min),
                "ranked_high_marapr_max_DU": (("pressure_range", "ranking_method", "rank_o3"), high_max),
                "ranked_high_marapr_mean_DU": (("pressure_range", "ranking_method", "rank_o3"), high_mean),
                "ranked_high_is_highest10": (("pressure_range", "ranking_method", "rank_o3"), high10),
                "ranked_high_is_highest25pct": (("pressure_range", "ranking_method", "rank_o3"), high25),
            },
            coords={"pressure_range": pressure_tags, "ranking_method": method_names, "rank_o3": rank_coord},
            attrs={
                "ranking_source": "CAM date variable (YYYYMMDD), with no-leap time fallback for missing date entries",
                "ranking_months": ",".join(str(m) for m in RANKING_MONTHS),
                "ranking_methods": ",".join(RANKING_METHOD_ORDER),
                "ranking_method_raw": "unsmoothed daily partial O3",
                "ranking_method_rm5": "5-day running mean daily partial O3",
                "ranking_valid_min_DU": float(RANKING_MIN_VALID_DU),
            },
        )
        ranking_ds.attrs["csv_rows"] = list(csv_records.values())
        return ranking_ds
    finally:
        ds.close()


def ranking_rows_from_dataset(case_name, ranking_ds):
    rows = list(ranking_ds.attrs.get("csv_rows", []))
    for row in rows:
        for col in RANKING_CSV_COLUMNS:
            row.setdefault(col, False if col.startswith("is_") else (-9999 if col.startswith("rank_") else np.nan))
    rows = sorted(rows, key=lambda r: (str(r["pressure_range"]), int(r["year"])))
    return rows


def write_ranking_csv(case_name, out_dir, ranking_ds):
    csv_path = ranking_csv_file(case_name, out_dir)
    rows = ranking_rows_from_dataset(case_name, ranking_ds)
    with csv_path.open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=RANKING_CSV_COLUMNS)
        writer.writeheader()
        writer.writerows(rows)
    print(f"[{case_name}] Saved raw+rm5 low/high ranking CSV: {csv_path}")
    return csv_path


def write_ranking_csv_from_aggregate(case_name, out_dir, aggregate_nc):
    ranking_ds = make_ranking_dataset_from_aggregate(case_name, aggregate_nc)
    try:
        return write_ranking_csv(case_name, out_dir, ranking_ds)
    finally:
        ranking_ds.close()


def append_ranking_to_aggregate_nc(case_name, aggregate_nc, overwrite=OVERWRITE):
    out_dir = Path(aggregate_nc).parent

    if RANKING_CSV_ONLY:
        print(f"[{case_name}] RANKING_CSV_ONLY=True: rebuilding raw+rm5 CSV only; NetCDF ranking variables unchanged.")
        write_ranking_csv_from_aggregate(case_name, out_dir, aggregate_nc)
        return aggregate_nc

    if aggregate_has_any_ranking_variables(aggregate_nc) and not overwrite:
        print(f"[{case_name}] Ranking-like variables already exist in {aggregate_nc}; skip NetCDF append.")
        print(f"[{case_name}] Recomputing raw+rm5 low/high ranking CSV from daily aggregate data.")
        write_ranking_csv_from_aggregate(case_name, out_dir, aggregate_nc)
        return aggregate_nc

    print(f"[{case_name}] Building raw+rm5 low/high ranking variables from aggregate NetCDF")
    ranking_ds = make_ranking_dataset_from_aggregate(case_name, aggregate_nc)
    try:
        print(f"[{case_name}] Appending ranking variables into: {aggregate_nc}")
        append_ds = ranking_ds.drop_vars("pressure_range") if "pressure_range" in ranking_ds.coords else ranking_ds
        if "csv_rows" in append_ds.attrs:
            append_ds = append_ds.copy()
            append_ds.attrs.pop("csv_rows", None)
        append_dataset_to_netcdf(append_ds, aggregate_nc)
        write_ranking_csv(case_name, out_dir, ranking_ds)
    finally:
        ranking_ds.close()
    return aggregate_nc


def process_case(case_name, case_dir, overwrite=OVERWRITE):
    case_dir = Path(case_dir)
    out_dir = case_dir / "partial_O3"
    out_dir.mkdir(parents=True, exist_ok=True)

    files = discover_o3_files(case_dir)
    if not files:
        raise FileNotFoundError(f"No O3 files found in {case_dir / 'O3'}")

    out_file = case_output_file(case_name, out_dir)

    print(f"\n===== {case_name} =====")
    print(f"Input O3 files: {len(files)}")
    print(f"Output dir: {out_dir}")
    print(f"Aggregate NetCDF: {out_file.name}")

    if RANKING_CSV_ONLY:
        if not is_valid_netcdf(out_file):
            msg = f"RANKING_CSV_ONLY=True but aggregate NetCDF is missing/unreadable: {out_file}"
            if RANKING_CSV_ONLY_SKIP_MISSING:
                print(f"[{case_name}] SKIP: {msg}")
                return {"aggregate_nc": out_file, "ranking_csv": None, "skipped": True}
            raise FileNotFoundError(msg)
        write_ranking_csv_from_aggregate(case_name, out_dir, out_file)
        return {"aggregate_nc": out_file, "ranking_csv": ranking_csv_file(case_name, out_dir)}

    if MAKE_AGGREGATE_NC:
        write_case_aggregate_nc(case_name, case_dir, files, out_file, overwrite=overwrite)

    if MAKE_RANKINGS:
        append_ranking_to_aggregate_nc(case_name, out_file, overwrite=overwrite)

    return {"aggregate_nc": out_file, "ranking_csv": ranking_csv_file(case_name, out_dir)}


def process_all_cases(overwrite=OVERWRITE):
    results = {}
    for case_name, case_dir in CASES.items():
        results[case_name] = process_case(case_name, case_dir, overwrite=overwrite)
    print("\nAll requested aggregate partial O3 calculations and NetCDF rankings are complete.")
    return results


In [5]:
# Run all cases. Set OVERWRITE=True in the config cell if you want to rebuild existing files.
results = process_all_cases(overwrite=OVERWRITE)



===== B2000WCN =====
Input O3 files: 210
Output dir: /mnt/soclim0/public_data/weiji/B2000WCN001002_timefixed/partial_O3
Aggregate NetCDF: B2000WCN_partial_O3_all_ranges.nc
[B2000WCN] 30_70hPa raw: skipped 3 years with insufficient valid O3 days
[B2000WCN] 30_70hPa raw lowest 10 event years: [ 98 158 168 197   3  40   8 110  54 102]
[B2000WCN] 30_70hPa raw highest 10 event years: [136  18 117  91  38 106  25 200 209 159]
[B2000WCN] 30_70hPa rm5: skipped 3 years with insufficient valid O3 days
[B2000WCN] 30_70hPa rm5 lowest 10 event years: [ 98 158 168 197   3  40 110   8  54 102]
[B2000WCN] 30_70hPa rm5 highest 10 event years: [136  18 117  91 106  38  25 200 209 156]
[B2000WCN] 1_100hPa raw: skipped 3 years with insufficient valid O3 days
[B2000WCN] 1_100hPa raw lowest 10 event years: [ 98 158 168 197   3  40  54   8 102 123]
[B2000WCN] 1_100hPa raw highest 10 event years: [136 106  38 124 117  25 200  18 161  91]
[B2000WCN] 1_100hPa rm5: skipped 3 years with insufficient valid O3 day

# ============================================================
# 重要 BUG 记录（MERRA2 pressure-level O3 柱含量积分）
# ------------------------------------------------------------
# 旧版本代码在 get_lev_weights() 里存在一个关键错位 bug：
#
#   1）先对 lev 做了排序，用排序后的 lev 去计算 layer edges 和 overlap
#   2）但最后构造 DataArray 时，overlap 数值没有恢复到原始 lev 顺序，
#      却直接绑定回原始 lev 坐标
#
# 这样会导致：
#   - overlap 数值对应的是“排序后的层”
#   - lev 标签对应的是“原始文件层顺序”
#   - xarray 按 lev 标签对齐后，层权重会贴到错误的 pressure level 上
#
# 后果：
#   名义上的“30–70 hPa 积分”实际上没有用对 70/50/40/30 hPa 的厚度，
#   而是把部分权重错配到了 20/10 hPa 等错误层，导致 partial O3 column
#   被系统性高估。
#
# 这个 bug 的典型表现是：
#   - 30/40/50/70 hPa 各层 O3 climatology 看起来和 WACCM 差别不大
#   - 但 30–70 hPa partial column 却离谱地偏高（可高出 ~40–50 DU）
#
# 正确修复方法：
#   - lev 只在内部临时排序用于计算 overlap
#   - overlap 计算完成后，必须恢复到原始 lev 顺序
#   - 保证 overlap 数值与 lev 坐标标签严格一一对应
#
# 另外：
#   - MERRA2 原始 O3 这里按 kg/kg 处理
#   - 在做 DU 积分前，必须先乘 (M_air / M_O3) 转成 mol/mol
#
# 经验教训：
#   任何涉及 xarray 坐标对齐的权重计算，只要中途做过 sort/reindex，
#   都必须检查“权重数值”和“坐标标签”是否仍然严格对应。
#   否则结果可能数值上看起来合理，但物理上完全错层。
# ============================================================
import os
import glob
import numpy as np
import xarray as xr
from tqdm import tqdm
from joblib import Parallel, delayed

# ============================================================
# 1. 基础配置与路径
# ============================================================
DATA_DIR_O3 = "/mnt/soclim0/public_data/weiji/MERRA2_Processed/O3"
OUT_ROOT    = "/home/weiji/restart_exam/code/20260210EXTR_B2000WCN_BWCN_COMPARE/result/data"
EXP_NAME    = "MERRA2"

N_CORES = 10  # 建议根据磁盘 I/O 适当调整，通常 8-12 较好

PRESSURE_RANGES = [
    (30, 70,  "30_70hPa"),
    (1,  100, "1_100hPa"),
]

LAT_BAND = (60, 90)
AUTO_ENSURE_DAILY = True

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

# ============================================================
# 2. 物理常数
# ============================================================
G      = 9.80665
M_AIR  = 28.964e-3
M_O3   = 47.9982e-3
NA     = 6.02214e23
DU_DEN = 2.687e20

# MERRA2 原始 O3: kg/kg -> mol/mol
KGKG_TO_MOLMOL = M_AIR / M_O3

# mol/mol -> DU
DU_FACTOR = NA / (G * M_AIR * DU_DEN)

# ============================================================
# 3. 修复后的 pressure-level overlap 权重
# ============================================================
def get_lev_weights(lev_hpa, p_top_hpa, p_bot_hpa):
    """
    对固定 pressure-level 数据：
    用 level center 构造 layer edges，
    再计算每层与目标 pressure range 的 overlap (Pa)。

    修复点：
    先按排序后的 lev 算 overlap，再还原回原始 lev 顺序，
    保证 overlap 数值与 lev 坐标严格一一对应。
    """
    lev_orig = np.asarray(lev_hpa, dtype=float)

    # 升序排序用于计算 layer edges
    sort_idx = np.argsort(lev_orig)
    lev_sorted = lev_orig[sort_idx]

    logp = np.log(lev_sorted)
    mids = np.exp(0.5 * (logp[:-1] + logp[1:]))

    edges = np.empty(lev_sorted.size + 1)
    edges[1:-1] = mids
    edges[0]  = lev_sorted[0]**2 / mids[0]
    edges[-1] = lev_sorted[-1]**2 / mids[-1]

    # hPa -> Pa
    p_layer_top = edges[:-1] * 100.0
    p_layer_bot = edges[1:]  * 100.0
    pT = float(p_top_hpa) * 100.0
    pB = float(p_bot_hpa) * 100.0

    overlap_sorted = np.maximum(
        0.0,
        np.minimum(p_layer_bot, pB) - np.maximum(p_layer_top, pT)
    )

    # 还原到原始 lev 顺序
    overlap_orig = np.empty_like(overlap_sorted)
    overlap_orig[sort_idx] = overlap_sorted

    return xr.DataArray(overlap_orig, coords={"lev": lev_hpa}, dims=("lev",))

# ============================================================
# 4. 单文件处理
# ============================================================
def process_single_year_file(file_path, weight_dict):
    try:
        with xr.open_dataset(file_path, decode_times=True) as ds:
            # 空间裁剪；不再强制 sortby("lev")，因为权重已和原始 lev 对齐
            da_o3 = ds["O3"].sel(lat=slice(LAT_BAND[0], LAT_BAND[1]))

            # MERRA2 原始 O3 是 kg/kg，直接强制转 mol/mol
            da_o3 = da_o3 * KGKG_TO_MOLMOL

            # 纬度权重
            weights_lat = np.cos(np.deg2rad(da_o3.lat))

            year_results = {}
            for tag, lev_w in weight_dict.items():
                # 垂直积分：sum( O3_molmol * overlap_Pa ) * DU_FACTOR
                column = (da_o3 * lev_w).sum(dim="lev") * DU_FACTOR

                # 极区平均
                ts = column.weighted(weights_lat).mean(dim=["lat", "lon"]).compute()
                year_results[tag] = ts

            return year_results

    except Exception as e:
        return f"Error {os.path.basename(file_path)}: {e}"

# ============================================================
# 5. 主程序
# ============================================================
def run_main():
    exp_out_dir = os.path.join(OUT_ROOT, EXP_NAME)
    os.makedirs(exp_out_dir, exist_ok=True)

    all_files = sorted(glob.glob(os.path.join(DATA_DIR_O3, "MERRA2.O3.*.nc")))
    if not all_files:
        print(f"No files found in {DATA_DIR_O3}")
        return

    # 读一个样例文件获取 lev 坐标
    with xr.open_dataset(all_files[0]) as ds_tmp:
        lev_coords = ds_tmp["lev"].values

    print("[INFO] Raw lev coordinates:")
    print(lev_coords)

    # 预计算各 pressure range 的 overlap 权重
    weight_dict = {
        tag: get_lev_weights(lev_coords, ptop, pbot)
        for ptop, pbot, tag in PRESSURE_RANGES
    }

    # 打印非零权重层，方便自检
    for ptop, pbot, tag in PRESSURE_RANGES:
        w = weight_dict[tag]
        nz = w.where(w > 0, drop=True)
        print(f"\n[INFO] Nonzero weights for {tag} ({ptop}-{pbot} hPa):")
        print(nz)

    print(f"\n🚀 开始并行处理 MERRA2 O3 ({N_CORES} cores)...")
    results = Parallel(n_jobs=N_CORES)(
        delayed(process_single_year_file)(f, weight_dict)
        for f in tqdm(all_files)
    )

    clean_results = [r for r in results if isinstance(r, dict)]
    error_results = [r for r in results if isinstance(r, str)]

    if error_results:
        print("\n[WARN] Some files failed:")
        for msg in error_results:
            print("  ", msg)

    if not clean_results:
        print("[ERROR] No valid results generated.")
        return

    # ========================================================
    # 汇总输出
    # ========================================================
    for _, _, tag in PRESSURE_RANGES:
        print(f"\n📦 正在汇总 {tag}...")

        combined = xr.concat([r[tag] for r in clean_results], dim="time").sortby("time")

        if AUTO_ENSURE_DAILY:
            # 若同一天有多个时次，自动 daily mean
            if len(np.unique(combined.time.dt.date)) < len(combined.time):
                combined = combined.resample(time="1D").mean()

        # 保存 nc
        out_nc = os.path.join(exp_out_dir, f"O3_pc_{EXP_NAME}_{tag}.nc")
        combined.name = "O3_column"
        combined.attrs = {
            "units": "DU",
            "lat_band": f"{LAT_BAND[0]}-{LAT_BAND[1]}N",
            "source_o3_unit": "kg/kg",
            "converted_to": "mol/mol before column integration",
            "method": "pressure-level overlap integration (fixed bug)",
            "pressure_range": tag
        }
        combined.to_netcdf(out_nc)
        print(f"  Saved: {out_nc}")

        # ====================================================
        # 极值分析：3–4 月最小值
        # ====================================================
        o3_5d = combined.rolling(time=5, center=True, min_periods=5).mean()
        spring = o3_5d.sel(time=o3_5d.time.dt.month.isin([3, 4]))
        spring_min = spring.groupby("time.year").min()

        # 过滤数据不足 58 天的年份
        counts = spring.groupby("time.year").count()
        valid_years = counts.year.where(counts >= 58, drop=True)
        spring_min = spring_min.sel(year=valid_years)

        sorted_years = spring_min.sortby(spring_min).year.values

        lowest10_txt = os.path.join(exp_out_dir, f"lowest10_years_sorted_{tag}.txt")
        np.savetxt(lowest10_txt, sorted_years[:10], fmt="%04d")
        print(f"  Saved: {lowest10_txt}")

        n_25pct = max(1, int(0.25 * len(sorted_years)))
        low25_txt = os.path.join(exp_out_dir, f"low25pct_years_{tag}.txt")
        np.savetxt(low25_txt, sorted_years[:n_25pct], fmt="%04d")
        print(f"  Saved: {low25_txt}")

        print(f"  Mean DU = {float(combined.mean().values):.3f}")
        print(f"  Min  DU = {float(combined.min().values):.3f}")
        print(f"  Max  DU = {float(combined.max().values):.3f}")

if __name__ == "__main__":
    run_main()